In [3]:
import pandas as pd

In [4]:
dataset = pd.read_csv("target_tabungan.csv")

In [5]:
dataset.head()

,tanggal,nama_target_id,nama_target,jumlah_target,nabung_harian,jumlah_terkumpul,progres,sisa_target,status_id,status
0,01/01/2022,0,Iphone 17 Pro Max,25000000,300000,300000,0.01,0.99,0,BELUM TERPENUHI
1,01/02/2022,0,Iphone 17 Pro Max,25000000,500000,800000,0.03,0.97,0,BELUM TERPENUHI
2,01/03/2022,0,Iphone 17 Pro Max,25000000,200000,1000000,0.04,0.96,0,BELUM TERPENUHI
3,01/04/2022,0,Iphone 17 Pro Max,25000000,500000,1500000,0.06,0.94,0,BELUM TERPENUHI
4,01/05/2022,0,Iphone 17 Pro Max,25000000,500000,2000000,0.08,0.92,0,BELUM TERPENUHI


In [6]:
dataset['tanggal'] = pd.to_datetime(dataset['tanggal'])

finish_dates = (
    dataset[dataset['status_id'] == 1]
    .groupby('nama_target_id')['tanggal']
    .max()
)

dataset['finish_date'] = dataset['nama_target_id'].map(finish_dates)

dataset['remaining_days'] = (
    dataset['finish_date'] - dataset['tanggal']
).dt.days

df = dataset[['tanggal', 'jumlah_target', 'nabung_harian', 'jumlah_terkumpul', 'status_id', 'remaining_days']].copy()
df

,tanggal,jumlah_target,nabung_harian,jumlah_terkumpul,status_id,remaining_days
0,2022-01-01,25000000,300000,300000,0,106
1,2022-01-02,25000000,500000,800000,0,105
2,2022-01-03,25000000,200000,1000000,0,104
3,2022-01-04,25000000,500000,1500000,0,103
4,2022-01-05,25000000,500000,2000000,0,102
...,...,...,...,...,...,...
1206,2025-04-21,8000000,300000,6750000,0,4
1207,2025-04-22,8000000,200000,6950000,0,3
1208,2025-04-23,8000000,100000,7050000,0,2
1209,2025-04-24,8000000,500000,7550000,0,1


In [7]:
from sklearn.model_selection import train_test_split

X = df[
    [
        'jumlah_target',
        'nabung_harian',
        'jumlah_terkumpul'
    ]
]

y = df['remaining_days']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [9]:
import tensorflow as tf
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Model

inputs = tf.keras.Input(shape=(X_train.shape[1],))

x = Dense(128, activation='relu')(inputs)
x = Dense(64, activation='relu')(x)
x = Dense(32, activation='relu')(x)

outputs = Dense(1)(x)

model = Model(inputs, outputs)

model.compile(
    optimizer='adam',
    loss='mae',
    metrics=['mae']
)

In [10]:
class StopAtMAE(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        mae = logs.get('mae')

        if mae < 0.03:
            self.model.stop_training = True

In [11]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[StopAtMAE()]
)

Epoch 1/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 55.1739 - mae: 55.1739 - val_loss: 56.2602 - val_mae: 56.2602
Epoch 2/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 51.6995 - mae: 51.6995 - val_loss: 49.3127 - val_mae: 49.3127
Epoch 3/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 43.0661 - mae: 43.0661 - val_loss: 36.0344 - val_mae: 36.0344
Epoch 4/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 28.8944 - mae: 28.8944 - val_loss: 17.1713 - val_mae: 17.1713
Epoch 5/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 15.7398 - mae: 15.7398 - val_loss: 11.0434 - val_mae: 11.0434
Epoch 6/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 10.2394 - mae: 10.2394 - val_loss: 8.5622 - val_mae: 8.5622
Epoch 7/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 7.9779 - mae: 7.9779 - val_loss: 7.1931 - val_mae: 7.1931
Epoch 8/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 6.5280 - mae: 6.5280 - val_loss: 5.9466 - val_mae: 5.9466
Epoch 9/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 

In [12]:
loss, mae = model.evaluate(X_test, y_test)

print("MAE:", mae)

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.8188 - mae: 2.8188 
MAE: 2.818781614303589


In [13]:
pred = model.predict(X_test)

comparison = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': pred.flatten()
})

comparison.head(10)

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 


,Actual,Predicted
0,5,4.662065
1,131,130.056992
2,19,21.232552
3,129,125.239754
4,30,30.884085
5,55,54.251263
6,21,21.391649
7,57,57.768658
8,38,35.864758
9,40,43.851906


In [14]:
from sklearn.metrics import r2_score

r2 = r2_score(y_test, pred)

print("R2 Score:", r2)

R2 Score: 0.987911581993103


In [21]:
model.save("model_tabungan.keras")

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

shutil.copy('model_tabungan.keras', '/content/drive/MyDrive/model_tabungan.keras')

Mounted at /content/drive


'/content/drive/MyDrive/model_tabungan.keras'

Inference

In [18]:
new_data = pd.DataFrame({
    'jumlah_target': [28000000],
    'nabung_harian': [100000],
    'jumlah_terkumpul': [1000000],
})

new_data_scaled = scaler.transform(new_data)

prediction = model.predict(new_data_scaled)

prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step


array([[124.42208]], dtype=float32)